<a href="https://colab.research.google.com/github/KammariBhavani/Emotion-Prediction/blob/main/EmotionPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Required Libraries**

In [3]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.ensemble import RandomForestClassifier

# **LOAD THE DATA**

In [4]:
train = pd.read_excel("Sample_arvyax_reflective_dataset.xlsx")
test = pd.read_csv("arvyax_test_inputs_120.xlsx - Sheet1.csv")

In [5]:
train.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5


In [6]:
test.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality
0,10001,woke up feeling more organized mentally. i was...,cafe,4,8.5,3,1,night,mixed,happy_face,vague
1,10002,started off distracted most of the time. this ...,mountain,4,8.5,1,2,afternoon,mixed,happy_face,clear
2,10003,kinda calm ...,cafe,15,8.5,2,5,evening,calm,happy_face,vague
3,10004,after the session i felt able to think straigh...,ocean,7,7.0,2,3,morning,overwhelmed,none,clear
4,10005,lowkey felt pretty grounded. i had to restart ...,ocean,20,8.5,1,5,afternoon,calm,tired_face,vague


# **DATA CLEANING**

In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

train['journal_text'] = train['journal_text'].apply(clean_text)
test['journal_text'] = test['journal_text'].apply(clean_text)

# FEATURE ENGINEERING

In [8]:
#Feature Engineering - TF-IDF

tfidf = TfidfVectorizer(max_features=500)
X_train_text = tfidf.fit_transform(train['journal_text'])
X_test_text = tfidf.transform(test['journal_text'])

In [16]:
 #Convert categorical columns

cat_cols = ['ambience_type', 'time_of_day', 'previous_day_mood',
            'face_emotion_hint', 'reflection_quality']

train_cat = pd.get_dummies(train[cat_cols])
test_cat = pd.get_dummies(test[cat_cols])

In [17]:
#aligning columns

train_cat, test_cat = train_cat.align(test_cat, join='left', axis=1, fill_value=0)

In [18]:
 #Numerical features

num_cols = ['duration_min', 'sleep_hours', 'energy_level', 'stress_level']

train_num = train[num_cols]
test_num = test[num_cols]

In [19]:
#combining all features using hstack instead of concat

# Ensure categorical features are numeric (0s and 1s)
train_cat_numeric = train_cat.astype(int)
test_cat_numeric = test_cat.astype(int)

# Fill NaN values in numerical features (e.g., with 0)
train_num_filled = train_num.fillna(0)
test_num_filled = test_num.fillna(0)

X = hstack([X_train_text, train_cat_numeric, train_num_filled])
X_test = hstack([X_test_text, test_cat_numeric, test_num_filled])

# MODEL BUILDING

***Model 1***



In [20]:
#Model 1: Emotion Prediction

y_emotion = train['emotional_state']
emotion_model = RandomForestClassifier()
emotion_model.fit(X, y_emotion)

emotion_pred = emotion_model.predict(X_test)

***Model 2***

In [21]:
#Model 2: Intensity Prediction
from sklearn.ensemble import RandomForestRegressor
y_intensity = train['intensity']

model_intensity = RandomForestRegressor()
model_intensity.fit(X, y_intensity)

intensity_pred = model_intensity.predict(X_test)

# **Decision**

In [22]:
def recommend_action(emotion, intensity, stress, sleep, energy):

    if emotion in ['sad', 'anxious']:
        if intensity >= 4:
            return "Take a break and talk to someone", "immediately"
        else:
            return "Go for a walk", "within 1 hour"

    elif emotion == 'happy':
        return "Continue your current activity", "now"

    elif emotion == 'angry':
        return "Do breathing exercise", "immediately"

    return "Reflect quietly", "tonight"

In [23]:
actions = []
timings = []

for i in range(len(test)):
    action, timing = recommend_action(
        emotion_pred[i],
        intensity_pred[i],
        test['stress_level'][i],
        test['sleep_hours'][i],
        test['energy_level'][i]
    )

    actions.append(action)
    timings.append(timing)

In [24]:
output = pd.DataFrame({
    'id': test['id'],
    'predicted_emotion': emotion_pred,
    'predicted_intensity': intensity_pred,
    'recommended_action': actions,
    'recommended_time': timings
})

output.to_csv("final_output.csv", index=False)

In [25]:
output.head(10)

,id,predicted_emotion,predicted_intensity,recommended_action,recommended_time
0,10001,focused,2.68,Reflect quietly,tonight
1,10002,restless,3.48,Reflect quietly,tonight
2,10003,focused,3.36,Reflect quietly,tonight
3,10004,focused,2.78,Reflect quietly,tonight
4,10005,restless,3.27,Reflect quietly,tonight
5,10006,mixed,3.25,Reflect quietly,tonight
6,10007,restless,3.72,Reflect quietly,tonight
7,10008,overwhelmed,3.28,Reflect quietly,tonight
8,10009,restless,2.88,Reflect quietly,tonight
9,10010,overwhelmed,3.26,Reflect quietly,tonight


In [26]:
from google.colab import files
files.download("final_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>